Этот код одновременно агрегирует слишком детальные и детализирует слишком агрегированные данные.

In [1]:
import pandas as pd
from collections import defaultdict

# Файлы
DATA_FILE = 'фз данные.xlsx'
CLASSIFIER_FILE = 'okved.xlsx'
OUTPUT_FILE = 'фз данные обработанные.xlsx'

df = pd.read_excel(DATA_FILE, converters={'OKVED': lambda x: str(x).strip()})
cls = pd.read_excel(CLASSIFIER_FILE, converters={1: lambda x: str(x).strip()})
cls = cls.rename(columns={cls.columns[1]: 'code'})

def normalize(code):
    parts = code.split('.')
    parts[0] = parts[0].zfill(2)
    return '.'.join(parts)

df['OKVED'] = df['OKVED'].apply(normalize)
cls['code'] = cls['code'].apply(normalize)

def to_okved3(code):
    parts = code.split('.')
    return parts[0] + '.' + parts[1][0]

value_cols = [c for c in df.columns if c not in ['OKVED', 'year']]

df['OKVED_3'] = df['OKVED'].apply(
    lambda x: to_okved3(x) if '.' in x else None
)

occupied_3 = set(df.loc[df['OKVED_3'].notna(), 'OKVED_3'])

codes_3 = sorted({
    to_okved3(c) for c in cls['code'] if '.' in c
})

map_2_to_3 = defaultdict(list)
for c in codes_3:
    map_2_to_3[c[:2]].append(c)

rows = []

for _, row in df.iterrows():
    code = row['OKVED']

    if '.' in code:
        new_row = row.copy()
        new_row['OKVED_3'] = to_okved3(code)
        rows.append(new_row)
    else:
        candidates = [
            c for c in map_2_to_3.get(code, [])
            if c not in occupied_3
        ]

        if not candidates:
            continue

        for c in candidates:
            new_row = row.copy()
            new_row['OKVED_3'] = c
            for col in value_cols:
                new_row[col] = new_row[col] / len(candidates)
            rows.append(new_row)

df_3 = (
    pd.DataFrame(rows)
    .groupby(['OKVED_3', 'year'], as_index=False)[value_cols]
    .sum()
)

df_3.to_excel(OUTPUT_FILE, index=False)